# ASR Evaluation: Base vs Fine-Tuned
Dataset: `gracecalista/new-dataset-transcribe`

> Warning
Evaluasi model dilakukan menggunakan versi library transformers yang berbeda untuk beberapa model.
Model Qwen3-ASR dijalankan menggunakan versi transformers yang lebih baru agar kompatibel dengan arsitektur model, sedangkan model Whisper dan MMS menggunakan versi transformers yang berbeda/stabil sesuai kebutuhan masing-masing model.

In [ ]:
import sys, importlib

sys.path.insert(0, "/root/.local/lib/python3.12/site-packages")

import huggingface_hub
_hf_api = importlib.import_module("huggingface_hub.hf_api")

# Patch 1: KernelInfo
if not hasattr(_hf_api, "KernelInfo"):
    from dataclasses import dataclass
    @dataclass
    class KernelInfo:
        id: str = None
    _hf_api.KernelInfo = KernelInfo
    huggingface_hub.KernelInfo = KernelInfo

# Patch 2: HfFolder
if not hasattr(_hf_api, "HfFolder"):
    class HfFolder:
        @staticmethod
        def get_token():
            return os.environ.get("HF_TOKEN", None)
    _hf_api.HfFolder = HfFolder
    huggingface_hub.HfFolder = HfFolder

# Patch 3: expose hf_api sebagai attribute (fix datasets/evaluate 4.x)
class _HfApiProxy:
    def __getattr__(self, name):
        return getattr(_hf_api, name)

huggingface_hub.hf_api = _HfApiProxy()

print(f"✅ Patch OK — huggingface_hub {huggingface_hub.__version__}")

# Cek Environment

In [ ]:
import subprocess, sys, os

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode == 0:
    print("GPU tersedia:")
    for line in gpu.stdout.split("\n"):
        if any(k in line for k in ["MiB", "T4", "P100", "A100"]):
            print(" ", line.strip())
else:
    print(" Tidak ada GPU — akan jalan di CPU (lambat)")

mem = subprocess.run(["free", "-h"], capture_output=True, text=True)
print("\n Memory:")
for line in mem.stdout.split("\n")[:2]:
    print(" ", line)

disk = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print("\n Disk /kaggle/working:")
for line in disk.stdout.split("\n")[:2]:
    print(" ", line)

# Login Hugging Face

In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print(" HF_TOKEN berhasil dimuat dari Kaggle Secrets")
except Exception as e:
    # Fallback: isi manual jika tidak pakai Secrets
    HF_TOKEN = "hf_xxx"   
    os.environ["HF_TOKEN"] = HF_TOKEN
    print(f"  Secrets tidak tersedia ({e})")
    print("   Pastikan HF_TOKEN diisi manual di cell ini")

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print(" Login Hugging Face berhasil")

# Install Dependencies

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + list(args),
        check=True
    )

pip(
    "transformers==4.57.6", # ← ganti ke versi yg kompatibel Whisper & MMS (4.51.0)
    "datasets==3.5.0",
    "evaluate==0.4.3",
    "jiwer==3.0.4",
    "accelerate==1.12.0",
    "soundfile",
    "peft",
    "qwen-asr" #← hapus jika tidak sedang menjalankan evaluasi qwen3
)

import transformers
print(transformers.__version__)

In [ ]:
import transformers, datasets, evaluate, torch
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"torch        : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Imports & Setup Path

In [ ]:
import csv, time, logging, warnings, zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
from tqdm.notebook import tqdm

from datasets import load_dataset, Audio, Dataset
from transformers import (
    AutoModelForCTC, AutoProcessor,
    WhisperForConditionalGeneration, WhisperProcessor,
)
import evaluate

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────
WORKING_DIR  = Path("/kaggle/working")
AUDIO_DIR    = WORKING_DIR / "wavs_extracted"  
LOG_FILE     = WORKING_DIR / "asr_eval.log"
CSV_PATH     = WORKING_DIR / "asr_resultz.csv"
PLOT_DIR     = WORKING_DIR / "plots"
PLOT_DIR.mkdir(exist_ok=True)
AUDIO_DIR.mkdir(exist_ok=True)

# ── HF Cache ke /kaggle/working ────────────────────────────────────
os.environ["HF_HOME"]            = str(WORKING_DIR / "hf_cache")
os.environ["TRANSFORMERS_CACHE"] = str(WORKING_DIR / "hf_cache/transformers")
os.environ["HF_DATASETS_CACHE"]  = str(WORKING_DIR / "hf_cache/datasets")
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(str(LOG_FILE), encoding="utf-8"),
    ],
)
logger = logging.getLogger(__name__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Device : {DEVICE.upper()}")
print(f" Output : {WORKING_DIR}")

# Konfigurasi Model

In [ ]:
@dataclass
class ModelConfig:
    model_name:   str
    model_type:   str
    repo_id:      str
    architecture: str
    language:     str = "id"
    task:         str = "transcribe"
    revision:     str = "main"
    subfolder:    str = ""

MODEL_CONFIGS = [
    # Whisper Large v3
    ModelConfig("Whisper Large v3", "base",      "openai/whisper-large-v3",                "whisper"),
    ModelConfig("Whisper Large v3", "finetuned", "gracecalista/whisper-khotbah-lora-v3",   "whisper",  subfolder="checkpoint-950"),

    # MMS
    ModelConfig("MMS",              "base",      "facebook/mms-1b-all",                    "mms",      language="ind"),
    ModelConfig("MMS",              "finetuned", "gracecalista/mms-khotbah-lora-final",    "mms",      language="ind", subfolder="checkpoint-950"),

    # Qwen3-ASR
    ModelConfig("Qwen3-ASR-0.6B", "base",     "Qwen/Qwen3-ASR-0.6B",                "qwen3asr", language="Indonesian"),
    ModelConfig("Qwen3-ASR-0.6B", "finetuned", "gracecalista/qwen3-asr-khotbah-lora-final", "qwen3asr", language="Indonesian", subfolder="last-checkpoint"),
]

# ── Parameter evaluasi ─────────────────────────────────────────────
HF_DATASET_REPO = "gracecalista/new-dataset-transcribe"
MAX_SAMPLES     = 5000   
BATCH_SIZE      = 4      

ACTIVE_MODELS = [c for c in MODEL_CONFIGS if c.model_name == "Qwen3-ASR-0.6B"] 
# ACTIVE_MODELS = [c for c in MODEL_CONFIGS if c.architecture in ("whisper", "mms")] #--> untuk whisper dan mms

# ── Qwen3 Scorer Config ────────────────────────────────────────────
QWEN_MODEL_ID    = "Qwen/Qwen3-0.6B"
QWEN_SCORE_SAMPLES = 5000   

print("Konfigurasi:")
for c in ACTIVE_MODELS:
    print(f"  [{c.model_type:9s}] {c.model_name:20s} → {c.repo_id}")
print(f"\n Qwen3 Scorer  : {QWEN_MODEL_ID} (scoring {QWEN_SCORE_SAMPLES} sampel/model)")


# Download & Ekstrak Dataset dari HF

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
import shutil

def download_and_extract_dataset(repo_id: str, extract_dir: Path, token: str, part: int = None):
    extract_dir.mkdir(parents=True, exist_ok=True)
    all_files = list(list_repo_files(repo_id, repo_type="dataset", token=token))
    zip_files = [f for f in all_files if f.startswith("wavs/") and f.endswith(".zip")]
    
    if part is not None:
        zip_files = [f for f in zip_files if f"part{part}" in Path(f).name or f"part_{part}" in Path(f).name]
        if not zip_files:
            print(f"  Tidak ada zip dengan 'part{part}' ditemukan!")
            print(f"   File zip yang ada: {[Path(f).name for f in all_files if f.startswith('wavs/') and f.endswith('.zip')]}")
            return
    
    if not zip_files:
        print("  Tidak ada .zip ditemukan di folder wavs/")
        return
    
    print(f" Ditemukan {len(zip_files)} file zip: {[Path(f).name for f in zip_files]}")
    
    for zip_path in tqdm(zip_files, desc="Download & Ekstrak zip", unit="file"):
        zip_name = Path(zip_path).name
        marker   = extract_dir / f".done_{zip_name}"
        if marker.exists():
            logger.info(f"  Skip {zip_name} (sudah diekstrak)")
            continue
        try:
            local_zip = Path(hf_hub_download(
                repo_id   = repo_id,
                filename  = zip_path,
                repo_type = "dataset",
                local_dir = "/tmp",
                token     = token,
            ))
            logger.info(f"  Downloaded to: {local_zip}")
            with zipfile.ZipFile(local_zip, "r") as zf:
                zf.extractall(extract_dir)
            marker.touch()
            local_zip.unlink(missing_ok=True)
        except Exception as e:
            logger.error(f"  Gagal proses {zip_name}: {e}")
    
    wav_count = len(list(extract_dir.rglob("*.wav")))
    print(f"\n Ekstraksi selesai!")
    print(f"   Total .wav ditemukan : {wav_count}")
    print(f"   Lokasi               : {extract_dir}")

# Panggil khusus part 3 (data yang belum pernah di fine-tuning)
download_and_extract_dataset(HF_DATASET_REPO, AUDIO_DIR, HF_TOKEN, part=3)

# Load metadata.csv & Bangun Dataset

In [ ]:
import csv

# Download metadata.csv
meta_local = hf_hub_download(
    repo_id   = HF_DATASET_REPO,
    filename  = "metadata.csv",
    repo_type = "dataset",
    token     = HF_TOKEN,
)
df_meta = pd.read_csv(meta_local, on_bad_lines="skip", engine="python", quotechar='"', sep=",")
print(f"✅ metadata.csv dimuat: {len(df_meta)} baris")
print(f"   Kolom: {list(df_meta.columns)}")
display(df_meta.head(3))

In [ ]:
# ── Auto-detect nama kolom ────────────────────────────────────────
FILE_COL = next(
    (c for c in df_meta.columns if any(k in c.lower() for k in ["file", "path", "name", "audio"])),
    df_meta.columns[0]
)
# Cari kolom teks/transcript
TEXT_COL = next(
    (c for c in df_meta.columns if any(k in c.lower() for k in ["transcript", "sentence", "text", "label"])),
    df_meta.columns[1]
)
print(f"✅ Kolom terdeteksi:")
print(f"   Audio/file  : '{FILE_COL}'")
print(f"   Transcript  : '{TEXT_COL}'")

In [ ]:
from tqdm import tqdm

print(" Membangun index audio...")
audio_index = {}
wav_files = list(AUDIO_DIR.rglob("*.wav"))
for f in tqdm(wav_files, desc="Indexing .wav", unit="file"):
    audio_index[f.name] = str(f)
print(f"    {len(audio_index)} file .wav terindeks")

def resolve_audio_path(file_name: str) -> Optional[str]:
    name = Path(file_name).name
    return audio_index.get(name)

print("\n Mencocokkan file_name dengan path audio...")
tqdm.pandas(desc="Resolving paths")
df_meta["resolved_path"] = df_meta[FILE_COL].progress_apply(
    lambda fn: resolve_audio_path(str(fn))
)

found   = df_meta["resolved_path"].notna().sum()
missing = df_meta["resolved_path"].isna().sum()
print(f"\n Cocok        : {found}")
print(f"  Tidak ketemu : {missing}")

if missing > 0:
    print("\nContoh yang tidak ketemu:")
    display(df_meta[df_meta["resolved_path"].isna()][[FILE_COL]].head(5))

df_valid = df_meta.dropna(subset=["resolved_path"]).reset_index(drop=True)
if MAX_SAMPLES:
    df_valid = df_valid.head(MAX_SAMPLES)

print(f"\n Dataset final: {len(df_valid)} sampel siap dievaluasi")

In [ ]:
# ── Bangun HF Dataset object ──────────────────────────────────────
ds = Dataset.from_dict({
    "audio"      : df_valid["resolved_path"].tolist(),
    "transcription": df_valid[TEXT_COL].tolist(),
})
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

AUDIO_COL = "audio"
LABEL_COL = "transcription"

print(f" Dataset siap: {len(ds)} sampel")
print(f"   Contoh teks[0]: {ds[0][LABEL_COL][:80]}")

# CSV Helper

In [ ]:
CSV_COLUMNS = [
    "model_name", "type", "wer",
    "total_insertions", "total_deletions", "total_substitutions",
    "pct_insertions", "pct_deletions", "pct_substitutions",
    "avg_inference_time", "total_samples", "qwen_score"
]

def init_csv(path=CSV_PATH):
    if not Path(path).exists():
        with open(path, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=CSV_COLUMNS, quoting=csv.QUOTE_ALL).writeheader()

def append_result_to_csv(result, path=CSV_PATH):
    with open(path, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=CSV_COLUMNS, quoting=csv.QUOTE_ALL).writerow(result)

init_csv()
print(f" CSV siap: {CSV_PATH}")


In [ ]:
from huggingface_hub import HfApi, upload_file
import json

CHECKPOINT_REPO = "gracecalista/asr-eval-cp" 
CHECKPOINT_FILE = WORKING_DIR / "checkpoint_state.json"
api = HfApi()

def save_checkpoint_to_hf(completed_configs: list):
    """Simpan progress ke HF Hub setelah tiap model selesai."""
    state = {
        "completed": [
            {"model_name": c.model_name, "model_type": c.model_type, "repo_id": c.repo_id}
            for c in completed_configs
        ]
    }
    
    CHECKPOINT_FILE.write_text(json.dumps(state, indent=2))
    
    # Upload CSV hasil + state ke HF
    try:
        api.create_repo(CHECKPOINT_REPO, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
        for local_file, remote_name in [
            (CSV_PATH,        "asr_results.csv"),
            (CHECKPOINT_FILE, "checkpoint_state.json"),
        ]:
            if local_file.exists():
                upload_file(
                    path_or_fileobj=str(local_file),
                    path_in_repo=remote_name,
                    repo_id=CHECKPOINT_REPO,
                    repo_type="dataset",
                    token=HF_TOKEN,
                )
        print(f"   Checkpoint tersimpan ke HF: {CHECKPOINT_REPO}")
    except Exception as e:
        print(f"    Gagal upload checkpoint: {e}")

def load_checkpoint_from_hf() -> set:
    """Load progress dari HF, return set of (model_name, model_type) yang sudah selesai."""
    try:
        path = hf_hub_download(
            repo_id=CHECKPOINT_REPO, filename="checkpoint_state.json",
            repo_type="dataset", token=HF_TOKEN, local_dir="/tmp"
        )
        state = json.loads(Path(path).read_text())
        done = {(c["model_name"], c["model_type"]) for c in state["completed"]}
        print(f"   Checkpoint ditemukan: {len(done)} model sudah selesai")
        return done
    except Exception:
        print("    Tidak ada checkpoint, mulai dari awal.")
        return set()

def restore_csv_from_hf():
    try:
        path = hf_hub_download(
            repo_id=CHECKPOINT_REPO, filename="asr_results.csv",
            repo_type="dataset", token=HF_TOKEN, local_dir="/tmp"
        )
        import shutil
        shutil.copy(path, CSV_PATH)
        print(f"   CSV lama direstorasi dari HF ({pd.read_csv(CSV_PATH).shape[0]} baris)")
    except Exception:
        print("    Tidak ada CSV lama, mulai fresh.")

restore_csv_from_hf()
init_csv()  
print(" Checkpoint system siap.")

# Model Loaders

In [ ]:
from peft import PeftModel

BASE_REPOS = {
    "whisper": "openai/whisper-large-v3",
    "mms":     "facebook/mms-1b-all",
}

def load_whisper(cfg, device):
    logger.info(f"  Loading Whisper: {cfg.repo_id}")
    processor_repo = BASE_REPOS["whisper"] if cfg.model_type == "finetuned" else cfg.repo_id
    processor = WhisperProcessor.from_pretrained(processor_repo, token=HF_TOKEN)
    dtype = torch.float16 if "cuda" in device else torch.float32

    if cfg.model_type == "finetuned":
        base = WhisperForConditionalGeneration.from_pretrained(
            BASE_REPOS["whisper"], torch_dtype=dtype, token=HF_TOKEN
        ).to(device)
        peft_model = PeftModel.from_pretrained(
            base, cfg.repo_id, subfolder=cfg.subfolder, token=HF_TOKEN
        )
        model = peft_model.merge_and_unload().eval()  # ← merge LoRA ke base
        logger.info("   LoRA Whisper di-merge ke base model")
    else:
        model = WhisperForConditionalGeneration.from_pretrained(
            cfg.repo_id, torch_dtype=dtype, token=HF_TOKEN
        ).to(device).eval()

    return model, processor


def load_mms(cfg, device, language="ind"):
    logger.info(f"  Loading MMS: {cfg.repo_id}")
    processor_repo = BASE_REPOS["mms"] if cfg.model_type == "finetuned" else cfg.repo_id
    processor = AutoProcessor.from_pretrained(processor_repo, token=HF_TOKEN)

    if hasattr(processor, "tokenizer") and hasattr(processor.tokenizer, "set_target_lang"):
        try:
            processor.tokenizer.set_target_lang(language)
        except Exception as e:
            logger.warning(f"  Tidak bisa set bahasa MMS '{language}': {e}")

    if cfg.model_type == "finetuned":
        base = AutoModelForCTC.from_pretrained(
            BASE_REPOS["mms"], token=HF_TOKEN
        ).to(device)
        try:
            base.load_adapter(language)
        except Exception as e:
            logger.warning(f"  Tidak bisa load adapter MMS '{language}': {e}")
        peft_model = PeftModel.from_pretrained(
            base, cfg.repo_id, subfolder=cfg.subfolder, token=HF_TOKEN
        )
        model = peft_model.merge_and_unload().eval()  
        logger.info("   LoRA MMS di-merge ke base model")
    else:
        model = AutoModelForCTC.from_pretrained(
            cfg.repo_id, token=HF_TOKEN
        ).to(device).eval()
        try:
            model.load_adapter(language)
        except Exception as e:
            logger.warning(f"  Tidak bisa load adapter MMS '{language}': {e}")

    return model, processor

def load_qwen3asr(cfg, device):
    import subprocess, sys, importlib

    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.57.6", "accelerate==1.12.0"], check=True)

    import transformers
    importlib.reload(transformers)

    from qwen_asr import Qwen3ASRModel
    from qwen_asr.core.transformers_backend.processing_qwen3_asr import Qwen3ASRProcessor
    from peft import PeftModel

    logger.info(f"  Loading Qwen3-ASR: {cfg.repo_id}")

    from huggingface_hub import snapshot_download
    local_model_dir = snapshot_download(
        repo_id="Qwen/Qwen3-ASR-0.6B",
        token=HF_TOKEN,
        ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
    )

    processor = Qwen3ASRProcessor.from_pretrained(
        local_model_dir,          # ← load dari path lokal, bukan repo ID
        trust_remote_code=True,
        local_files_only=True,    # ← tidak ada network call → tidak ada 404
    )
    qwen_wrapper = Qwen3ASRModel.from_pretrained(
        "Qwen/Qwen3-ASR-0.6B",
        dtype=torch.bfloat16,
        device_map=device,
    )
    inner = qwen_wrapper.model

    if cfg.model_type == "finetuned":
        peft_thinker = PeftModel.from_pretrained(
            inner.thinker, cfg.repo_id, is_trainable=False,
        )
        inner.thinker = peft_thinker.merge_and_unload()
        inner.thinker.eval()
        logger.info("   LoRA Qwen3-ASR thinker di-merge ke base model")
    else:
        inner.thinker.eval()

    return qwen_wrapper, processor


def load_model(config, device):
    arch = config.architecture.lower()
    if arch == "whisper":
        return load_whisper(config, device)
    elif arch == "mms":
        return load_mms(config, device, language=config.language)
    elif arch == "qwen3asr":
        return load_qwen3asr(config, device)
    else:
        raise ValueError(f"Arsitektur tidak dikenal: {arch}")

print("✅ Loader siap.")

# Qwen3-0.6B Scorer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, re

_qwen_model     = None
_qwen_tokenizer = None

def load_qwen_scorer(device):
    """Load Qwen3-0.6B sekali, cache global."""
    global _qwen_model, _qwen_tokenizer
    
    if _qwen_model is not None:
        return _qwen_model, _qwen_tokenizer
    print(f"   Loading Qwen3-0.6B scorer...")
    _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, token=HF_TOKEN)
    _qwen_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_ID,
        torch_dtype=torch.float16 if "cuda" in device else torch.float32,
        token=HF_TOKEN,
    ).to(device).eval()
    print(f"   Qwen3-0.6B siap di {device.upper()}")
    return _qwen_model, _qwen_tokenizer

def unload_qwen_scorer():
    """Hapus Qwen dari VRAM setelah scoring selesai."""
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is not None:
        del _qwen_model, _qwen_tokenizer
        _qwen_model = _qwen_tokenizer = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        print("    Qwen3 diturunkan dari VRAM")

def qwen_score_transcriptions(refs, preds, device, n_samples=None):
    """
    Beri skor 0–10 untuk setiap pasang (referensi, prediksi) pakai Qwen3-0.6B.
    Return rata-rata skor (float) atau None jika gagal.
    
    Qwen diminta menilai:
      - Kemiripan makna (semantic similarity)
      - Kelancaran teks prediksi
      - Kesesuaian konteks Bahasa Indonesia
    Skala 0 (sangat buruk) – 10 (sempurna).
    """
    if n_samples:
        idx   = list(range(min(n_samples, len(refs))))
        refs  = [refs[i]  for i in idx]
        preds = [preds[i] for i in idx]

    model, tokenizer = load_qwen_scorer(device)
    scores = []
    for ref, pred in tqdm(zip(refs, preds), total=len(refs),
                          desc="  Qwen3 scoring", unit="sample"):
        prompt = (
            "Kamu adalah penilai kualitas transkripsi otomatis (ASR) untuk Bahasa Indonesia.\n"
            "Berikan skor dari 0 sampai 10 untuk prediksi transkripsi berikut:\n"
            f"  Referensi : {ref}\n"
            f"  Prediksi  : {pred}\n\n"
            "Kriteria penilaian:\n"
            "  • Kemiripan makna dengan referensi\n"
            "  • Kelancaran dan keterbacaan teks prediksi\n"
            "  • Kesesuaian ejaan Bahasa Indonesia\n\n"
            "Jawab HANYA dengan satu angka bulat (0–10), tanpa penjelasan."
        )
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        inputs = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        m = re.search(r"\b(\d{1,2})\b", response)
        if m:
            s = int(m.group(1))
            scores.append(max(0, min(10, s)))   # clamp 0–10
        else:
            logger.warning(f"  Qwen parse gagal: '{response}' — skip sampel ini")

    if not scores:
        return None
    avg = round(sum(scores) / len(scores), 4)
    print(f"   Qwen3 avg score : {avg:.2f} / 10  (dari {len(scores)} sampel)")
    return avg

print(" Qwen3 scorer siap.")


# Fungsi Inferensi

In [ ]:
def transcribe_whisper(model, processor, audio_arrays, sr, device,
                       language="id", task="transcribe"):
    dtype = torch.float16 if "cuda" in device else torch.float32
    inputs = processor(audio_arrays, sampling_rate=sr,
                       return_tensors="pt", padding=True).input_features.to(device, dtype=dtype)
    with torch.no_grad():
        forced_ids = processor.get_decoder_prompt_ids(language=language, task=task)
        generated  = model.generate(inputs, forced_decoder_ids=forced_ids, max_new_tokens=225)
    return [t.strip() for t in processor.batch_decode(generated, skip_special_tokens=True)]

def transcribe_ctc(model, processor, audio_arrays, sr, device):
    inputs = processor(audio_arrays, sampling_rate=sr, return_tensors="pt", padding=True)
    iv  = inputs.input_values.to(device)
    attn = inputs.get("attention_mask")
    if attn is not None:
        attn = attn.to(device)
    with torch.no_grad():
        logits = model(iv, attention_mask=attn).logits
    return [t.strip() for t in processor.batch_decode(torch.argmax(logits, dim=-1))]


def transcribe_qwen3asr(model, processor, audio_arrays, sr, device):
    """Transcribe satu per satu pakai Qwen3ASRModel.transcribe()"""
    import tempfile, soundfile as sf_lib, os
    results = []
    for audio in audio_arrays:
        try:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                sf_lib.write(tmp.name, audio, sr)
                tmp_path = tmp.name
            text = model.transcribe(tmp_path, language="Indonesian")
            if isinstance(text, list):
                text = text[0]
            # ASRTranscription object — ambil attribute text-nya
            if hasattr(text, "text"):
                text = text.text
            elif hasattr(text, "transcription"):
                text = text.transcription
            results.append(text.strip() if text else "")
        except Exception as e:
            logger.warning(f"  qwen3asr transcribe error: {e}")
            results.append("")
        finally:
            try:
                os.unlink(tmp_path)
            except:
                pass
    return results

def transcribe_batch(config, model, processor, audio_arrays, sr, device):
    try:
        if config.architecture.lower() == "whisper":
            return transcribe_whisper(model, processor, audio_arrays, sr, device,
                                      language=config.language, task=config.task)
        elif config.architecture.lower() == "qwen3asr":   
            return transcribe_qwen3asr(model, processor, audio_arrays, sr, device)
        else:
            return transcribe_ctc(model, processor, audio_arrays, sr, device)
    except Exception as e:
        logger.error(f"  Gagal inferensi: {e}")
        return [""] * len(audio_arrays)

print(" Fungsi inferensi siap.")

# Modul IDS — Insertion / Deletion / Substitution (Levenshtein)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODUL IDS (Insertion / Deletion / Substitution) — Levenshtein Word Alignment
# ─────────────────────────────────────────────────────────────────────────────
# Modul ini mengimplementasikan perhitungan kesalahan ASR secara lebih granular
# dibanding WER biasa, dengan memisahkan tipe kesalahan menjadi:
#   • Insertion  (I): kata prediksi yang tidak ada di referensi
#   • Deletion   (D): kata referensi yang hilang dari prediksi
#   • Substitution (S): kata referensi yang diganti kata lain di prediksi
#
# Pendekatan: dynamic programming Levenshtein distance di level kata,
# identik dengan algoritma yang digunakan jiwer & NIST sclite.
# ─────────────────────────────────────────────────────────────────────────────

from typing import Tuple, List

def levenshtein_word_align(
    reference: List[str],
    hypothesis: List[str]
) -> Tuple[int, int, int, List[str]]:
    """
    Hitung Insertion, Deletion, Substitution menggunakan Levenshtein
    word-level alignment dengan backtracking.

    Parameter
    ---------
    reference  : list kata referensi (ground-truth)
    hypothesis : list kata hipotesis / prediksi ASR

    Return
    ------
    (insertions, deletions, substitutions, ops)
    ops : list operasi per posisi hasil backtrack
          'C' = Correct, 'S' = Substitution, 'D' = Deletion, 'I' = Insertion
    """
    n, m = len(reference), len(hypothesis)

    # ── Build DP table (cost matrix) ───────────────────────────────────────
    # dp[i][j] = edit distance antara reference[:i] dan hypothesis[:j]
    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i          # semua reference didelete
    for j in range(m + 1):
        dp[0][j] = j          # semua hypothesis diinsert

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if reference[i - 1] == hypothesis[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]          # match → cost 0
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],       # deletion  (hapus kata referensi)
                    dp[i][j - 1],       # insertion (tambah kata hipotesis)
                    dp[i - 1][j - 1],   # substitution
                )

    # ── Backtrack untuk mendapatkan urutan operasi ──────────────────────────
    ops = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and reference[i-1] == hypothesis[j-1]:
            ops.append('C')         # Correct
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            ops.append('S')         # Substitution
            i -= 1; j -= 1
        elif j > 0 and dp[i][j] == dp[i][j-1] + 1:
            ops.append('I')         # Insertion
            j -= 1
        else:
            ops.append('D')         # Deletion
            i -= 1

    ops.reverse()
    insertions    = ops.count('I')
    deletions     = ops.count('D')
    substitutions = ops.count('S')
    return insertions, deletions, substitutions, ops


def compute_ids_stats(
    references: List[str],
    hypotheses: List[str]
) -> dict:
    """
    Hitung total I/D/S dan persentasenya terhadap total kata referensi
    untuk seluruh dataset.

    Return dict berisi:
      total_ref_words, total_insertions, total_deletions, total_substitutions,
      pct_insertions, pct_deletions, pct_substitutions
    """
    total_I = total_D = total_S = total_ref = 0

    for ref_str, hyp_str in zip(references, hypotheses):
        ref_words = ref_str.strip().split()
        hyp_words = hyp_str.strip().split()
        I, D, S, _ = levenshtein_word_align(ref_words, hyp_words)
        total_I   += I
        total_D   += D
        total_S   += S
        total_ref += len(ref_words)

    pct_I = round(total_I / total_ref * 100, 4) if total_ref else 0.0
    pct_D = round(total_D / total_ref * 100, 4) if total_ref else 0.0
    pct_S = round(total_S / total_ref * 100, 4) if total_ref else 0.0

    return {
        "total_ref_words"    : total_ref,
        "total_insertions"   : total_I,
        "total_deletions"    : total_D,
        "total_substitutions": total_S,
        "pct_insertions"     : pct_I,
        "pct_deletions"      : pct_D,
        "pct_substitutions"  : pct_S,
    }


def print_ids_report(stats: dict, model_label: str = ""):
    """Tampilkan laporan I/D/S ke konsol dengan format tabel."""
    total_ref = stats["total_ref_words"]
    print(f"\n{'─'*56}")
    if model_label:
        print(f"  📊 IDS Report — {model_label}")
    print(f"{'─'*56}")
    print(f"  {'Metrik':<28}{'Jumlah':>10}{'% dari Ref':>12}")
    print(f"  {'─'*50}")
    for label, key_n, key_p in [
        ("Total Kata Referensi",  "total_ref_words",     None),
        ("Insertion  (I)",        "total_insertions",    "pct_insertions"),
        ("Deletion   (D)",        "total_deletions",     "pct_deletions"),
        ("Substitution (S)",      "total_substitutions", "pct_substitutions"),
    ]:
        n = stats[key_n]
        p = f"{stats[key_p]:.2f}%" if key_p else ""
        print(f"  {label:<28}{n:>10}{p:>12}")
    print(f"{'─'*56}")


def show_alignment_examples(
    references: List[str],
    hypotheses: List[str],
    n_examples: int = 5,
    only_errors: bool = True
):
    """
    Tampilkan contoh alignment Reference / Hypothesis / Operation.

    Parameter
    ---------
    n_examples  : jumlah contoh yang ditampilkan
    only_errors : jika True, hanya tampilkan sampel yang ada kesalahan (I/D/S)
    """
    shown = 0
    print(f"\n{'═'*60}")
    print("  CONTOH ALIGNMENT (Ref | Hyp | Op)")
    print(f"{'═'*60}")

    for idx, (ref_str, hyp_str) in enumerate(zip(references, hypotheses)):
        if shown >= n_examples:
            break
        ref_words = ref_str.strip().split()
        hyp_words = hyp_str.strip().split()
        I, D, S, ops = levenshtein_word_align(ref_words, hyp_words)

        if only_errors and (I + D + S) == 0:
            continue  # lewati sampel benar sempurna

        print(f"\n  Sampel #{idx + 1}  |  I={I}  D={D}  S={S}")
        print(f"  {'─'*56}")

        # Rekonstruksi tampilan per-kolom
        ref_display  = []
        hyp_display  = []
        ops_display  = []

        ri, hi = 0, 0
        for op in ops:
            if op == 'C':
                ref_display.append(ref_words[ri])
                hyp_display.append(hyp_words[hi])
                ops_display.append('C')
                ri += 1; hi += 1
            elif op == 'S':
                ref_display.append(ref_words[ri])
                hyp_display.append(hyp_words[hi])
                ops_display.append('S')
                ri += 1; hi += 1
            elif op == 'D':
                ref_display.append(ref_words[ri])
                hyp_display.append('***')
                ops_display.append('D')
                ri += 1
            elif op == 'I':
                ref_display.append('***')
                hyp_display.append(hyp_words[hi])
                ops_display.append('I')
                hi += 1

        # Print per baris (wrap setiap 8 token)
        chunk = 8
        for start in range(0, len(ops_display), chunk):
            sl = slice(start, start + chunk)
            r_row = ref_display[sl]
            h_row = hyp_display[sl]
            o_row = ops_display[sl]
            col_w = [max(len(r), len(h), 3) + 2 for r, h in zip(r_row, h_row)]
            print("  REF: " + "".join(w.ljust(c) for w, c in zip(r_row, col_w)))
            print("  HYP: " + "".join(w.ljust(c) for w, c in zip(h_row, col_w)))
            print("  OP:  " + "".join(o.ljust(c) for o, c in zip(o_row, col_w)))
            print()

        shown += 1

    if shown == 0:
        print("  (Tidak ada sampel dengan kesalahan ditemukan)")
    print(f"{'═'*60}")


print("✅ Modul IDS (Levenshtein word alignment) siap.")
print("   Fungsi tersedia: levenshtein_word_align, compute_ids_stats,")
print("                    print_ids_report, show_alignment_examples")


# Fungsi Evaluasi

In [ ]:
def evaluate_model(config, dataset, audio_col, text_col, device, batch_size=4):
    print(f"\n{'='*60}")
    print(f"▶  {config.model_name} [{config.model_type.upper()}]")
    print(f"   Repo : {config.repo_id}")
    try:
        model, processor = load_model(config, device)
    except Exception as e:
        logger.error(f"  GAGAL load model: {e}")
        result = {"model_name": config.model_name, "type": config.model_type,
                  "wer": None,
                  "total_insertions": None, "total_deletions": None, "total_substitutions": None,
                  "pct_insertions": None, "pct_deletions": None, "pct_substitutions": None,
                  "avg_inference_time": None, "total_samples": 0, "qwen_score": None}
        append_result_to_csv(result)
        return result

    if model is None or processor is None:
        logger.warning(f"  Skip {config.model_name} ({config.model_type}) — model/processor None")
        result = {"model_name": config.model_name, "type": config.model_type,
                  "wer": None,
                  "total_insertions": None, "total_deletions": None, "total_substitutions": None,
                  "pct_insertions": None, "pct_deletions": None, "pct_substitutions": None,
                  "avg_inference_time": None, "total_samples": 0, "qwen_score": None}
        append_result_to_csv(result)
        return result

    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"   VRAM : {used:.1f} / {total:.1f} GB")

    wer_metric      = evaluate.load("wer")
    all_refs, all_preds, inf_times = [], [], []
    total_samples = error_count = 0
    # ── IDS accumulators ──────────────────────────────────────
    ids_total_I = ids_total_D = ids_total_S = 0
    n = len(dataset)

    pbar = tqdm(range(0, n, batch_size),
                desc=f"  {config.model_name[:18]} [{config.model_type}]",
                unit="batch")

    for start in pbar:
        batch        = dataset.select(range(start, min(start + batch_size, n)))
        audio_arrays, refs = [], []
        for sample in batch:
            try:
                audio_arrays.append(sample[audio_col]["array"].astype(np.float32))
                refs.append(sample[text_col].strip().lower())
            except Exception as e:
                logger.warning(f"    Sampel corrupt: {e}")
                error_count += 1

        if not audio_arrays:
            continue

        t0    = time.perf_counter()
        preds = transcribe_batch(config, model, processor, audio_arrays, 16000, device)
        per_s = (time.perf_counter() - t0) / len(audio_arrays)
        inf_times.extend([per_s] * len(audio_arrays))
        all_refs.extend(refs)
        all_preds.extend([p.lower() for p in preds])
        total_samples += len(audio_arrays)

        cur_wer = wer_metric.compute(predictions=all_preds, references=all_refs)
        pbar.set_postfix({"WER": f"{cur_wer:.4f}", "ms/s": f"{per_s*1000:.0f}", "err": error_count})

    final_wer = wer_metric.compute(predictions=all_preds, references=all_refs) if all_preds else None
    avg_lat   = float(np.mean(inf_times)) if inf_times else None

    # ── Hitung IDS (Insertion / Deletion / Substitution) ─────────────────
    # compute_ids_stats melakukan Levenshtein word alignment untuk semua
    # pasang (referensi, prediksi) dan mengakumulasi total I, D, S.
    ids_stats = compute_ids_stats(all_refs, all_preds) if all_preds else None
    if ids_stats:
        ids_total_I = ids_stats["total_insertions"]
        ids_total_D = ids_stats["total_deletions"]
        ids_total_S = ids_stats["total_substitutions"]

    print(f"  ✓ WER      : {final_wer:.4f}" if final_wer else "  ✗ WER: N/A")
    print(f"  ✓ Latency  : {avg_lat*1000:.2f} ms/sample" if avg_lat else "  ✗ Latency: N/A")
    print(f"  ✓ Valid    : {total_samples}  |  corrupt: {error_count}")

    # ── Tampilkan laporan IDS di konsol ───────────────────────────────────
    if ids_stats:
        print_ids_report(ids_stats, model_label=f"{config.model_name} [{config.model_type.upper()}]")
        # Tampilkan 5 contoh alignment (hanya sampel yang ada error)
        show_alignment_examples(all_refs, all_preds, n_examples=5, only_errors=True)

    # ── Bebaskan VRAM ASR sebelum Qwen naik ───────────────────────
    del model, processor
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    # ── Qwen3-0.6B Scoring ────────────────────────────────────────
    print(f"  🤖 Menjalankan Qwen3 scorer ({QWEN_SCORE_SAMPLES} sampel)...")
    qwen_avg = None
    try:
        qwen_avg = qwen_score_transcriptions(
            all_refs, all_preds, device, n_samples=QWEN_SCORE_SAMPLES
        )
    except Exception as e:
        logger.error(f"  Qwen3 scoring gagal: {e}")
    finally:
        unload_qwen_scorer()   # turunkan Qwen dari VRAM setelah selesai

    result = {
        "model_name"          : config.model_name,
        "type"                : config.model_type,
        "wer"                 : round(final_wer, 4) if final_wer else None,
        # ── IDS fields ────────────────────────────────────────────────
        "total_insertions"    : ids_stats["total_insertions"]    if ids_stats else None,
        "total_deletions"     : ids_stats["total_deletions"]     if ids_stats else None,
        "total_substitutions" : ids_stats["total_substitutions"] if ids_stats else None,
        "pct_insertions"      : ids_stats["pct_insertions"]      if ids_stats else None,
        "pct_deletions"       : ids_stats["pct_deletions"]       if ids_stats else None,
        "pct_substitutions"   : ids_stats["pct_substitutions"]   if ids_stats else None,
        # ─────────────────────────────────────────────────────────────
        "avg_inference_time"  : round(avg_lat, 6) if avg_lat else None,
        "total_samples"       : total_samples,
        "qwen_score"          : qwen_avg,
    }
    append_result_to_csv(result)
    return result

print(" Fungsi evaluasi siap.")


# Jalankan Evaluasi

In [ ]:
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

In [ ]:
results = []
completed_configs = []

# Load checkpoint dari HF
done_set = load_checkpoint_from_hf()
done_set.discard(("Qwen3-ASR-0.6B", "base"))
done_set.discard(("Qwen3-ASR-0.6B", "finetuned"))

for config in ACTIVE_MODELS:
    key = (config.model_name, config.model_type)
    
    if key in done_set:
        print(f"  ⏭️  Skip {config.model_name} [{config.model_type}] — sudah selesai")
        completed_configs.append(config)
        continue

    result = evaluate_model(
        config, ds, AUDIO_COL, LABEL_COL,
        device=DEVICE, batch_size=BATCH_SIZE
    )
    results.append(result)
    completed_configs.append(config)

    # ← Simpan checkpoint setelah tiap model selesai
    save_checkpoint_to_hf(completed_configs)

print("\n" + "="*60)
print("✅ Evaluasi selesai!")
df_results = pd.read_csv(CSV_PATH)
display(df_results)

# Visualisasi

In [ ]:
def plot_results(csv_path=CSV_PATH, output_dir=PLOT_DIR):
    df = pd.read_csv(csv_path).dropna(subset=["wer", "avg_inference_time"])
    if df.empty:
        print("  Tidak ada data valid untuk diplot.")
        return

    df["latency_s"] = df["avg_inference_time"]  # tetap dalam detik
    sns.set_theme(style="whitegrid")

    COLOR_BASE = "#6A8FC4"
    COLOR_FT   = "#C86B6B"

    # ── Plot per model ─────────────────────────────────────────────
    for model in df["model_name"].unique():
        subset    = df[df["model_name"] == model]
        safe_name = model.replace(" ", "_").replace("/", "-")

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(f"Komparasi Performa ASR — {model}", fontsize=14, fontweight="bold")

        for ax, col, title, ylabel, fmt in [
            (axes[0], "wer",       "Komparasi WER\n(Base vs Fine-Tuned)",              "Skor WER (↓)",         "{:.4f}"),
            (axes[1], "latency_s", "Komparasi Kecepatan Inferensi\n(Base vs Fine-Tuned)", "Total Waktu (Detik)", "{:.3f}s"),
        ]:
            bars = sns.barplot(
                data=subset, x="type", y=col,
                palette={"base": COLOR_BASE, "finetuned": COLOR_FT},
                order=["base", "finetuned"], ax=ax
            )
            ax.set_title(title, fontsize=11)
            ax.set_xlabel("")
            ax.set_ylabel(ylabel)
            ax.set_xticklabels(["Base", "Fine-Tuned"])

            for bar in ax.patches:
                h = bar.get_height()
                if h > 0:
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        h + ax.get_ylim()[1] * 0.01,
                        fmt.format(h),
                        ha="center", va="bottom", fontsize=11, fontweight="bold"
                    )

        plt.tight_layout()
        out = Path(output_dir) / f"{safe_name}_comparison.png"
        plt.savefig(out, dpi=150, bbox_inches="tight")
        plt.show()

    # ── Plot semua model (grouped) ─────────────────────────────────
    if len(df["model_name"].unique()) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle("Komparasi Performa ASR — Semua Model", fontsize=14, fontweight="bold")

        for ax, col, title, ylabel, fmt in [
            (axes[0], "wer",       "Komparasi WER\n(Base vs Fine-Tuned)",                "Skor WER (↓)",         "{:.4f}"),
            (axes[1], "latency_s", "Komparasi Kecepatan Inferensi\n(Base vs Fine-Tuned)", "Total Waktu (Detik)", "{:.3f}s"),
        ]:
            sns.barplot(
                data=df, x="model_name", y=col, hue="type",
                palette={"base": COLOR_BASE, "finetuned": COLOR_FT},
                hue_order=["base", "finetuned"], ax=ax
            )
            ax.set_title(title, fontsize=11)
            ax.set_xlabel("")
            ax.set_ylabel(ylabel)
            ax.tick_params(axis="x", rotation=15)
            ax.legend(title="Tipe Model", labels=["Base", "Fine-Tuned"])

            for bar in ax.patches:
                h = bar.get_height()
                if h > 0:
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        h + ax.get_ylim()[1] * 0.01,
                        fmt.format(h),
                        ha="center", va="bottom", fontsize=9, fontweight="bold"
                    )

        plt.tight_layout()
        plt.savefig(Path(output_dir) / "all_models_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()

plot_results()

# Ringkasan & Download Output

In [ ]:
df_final = pd.read_csv(CSV_PATH, on_bad_lines="skip", engine="python")
print("=" * 60)
print(" RINGKASAN HASIL EVALUASI ASR")
print(f"   Dataset: gracecalista/new-dataset-transcribe")
print("=" * 60)

for model_name in df_final["model_name"].unique():
    subset = df_final[df_final["model_name"] == model_name]
    print(f"\n🤖 {model_name}")
    for _, row in subset.iterrows():
        wer_s   = f"{row['wer']:.4f}"       if pd.notna(row['wer'])               else "N/A"
        lat_s   = f"{row['avg_inference_time']*1000:.2f} ms" if pd.notna(row['avg_inference_time']) else "N/A"
        qwen_s  = f"{row['qwen_score']:.2f}/10" if pd.notna(row.get('qwen_score')) else "N/A"
        print(f"   [{row['type']:9s}]  WER: {wer_s:8s}  Latency: {lat_s:10s}  Qwen Score: {qwen_s}")
        # ── Tampilkan IDS breakdown ──────────────────────────────────
        ids_cols = ["total_insertions", "total_deletions", "total_substitutions",
                    "pct_insertions", "pct_deletions", "pct_substitutions"]
        if all(c in row.index for c in ids_cols) and pd.notna(row.get('total_insertions')):
            print(f"              {'':9s}  I: {int(row['total_insertions']):>6}  "
                  f"D: {int(row['total_deletions']):>6}  "
                  f"S: {int(row['total_substitutions']):>6}")
            print(f"              {'':9s}  I: {row['pct_insertions']:>5.2f}%  "
                  f"D: {row['pct_deletions']:>5.2f}%  "
                  f"S: {row['pct_substitutions']:>5.2f}% (terhadap total kata ref)")
    b = subset[subset["type"]=="base"]["wer"].values
    f = subset[subset["type"]=="finetuned"]["wer"].values
    if len(b) and len(f) and pd.notna(b[0]) and pd.notna(f[0]) and b[0] > 0:
        imp  = (b[0] - f[0]) / b[0] * 100
        print(f"   {'✅' if imp > 0 else '⚠️ '} WER improvement: {imp:+.1f}%")
    # Qwen score comparison
    bq = subset[subset["type"]=="base"]["qwen_score"].values
    fq = subset[subset["type"]=="finetuned"]["qwen_score"].values
    if len(bq) and len(fq) and pd.notna(bq[0]) and pd.notna(fq[0]):
        delta = fq[0] - bq[0]
        print(f"   {'✅' if delta > 0 else '⚠️ '} Qwen score delta: {delta:+.2f}")

print("\n Output files (tab Output → klik untuk download):")
print(f"    {CSV_PATH.name}")
for p in sorted(PLOT_DIR.glob("*.png")):
    print(f"    plots/{p.name}")
print(f"    {LOG_FILE.name}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# GRAFIK IDS GABUNGAN SEMUA MODEL
# ─────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load CSV hasil evaluasi
df = pd.read_csv(CSV_PATH)

# Nama error
metrics = ["Insertion", "Deletion", "Substitution"]

# Ambil model unik
models = df["model_name"].unique()

# Posisi x
x = np.arange(len(models))

# Lebar bar
width = 0.12

plt.figure(figsize=(14, 6))

colors = {
    "base_insert": "#4E79A7",
    "ft_insert": "#A0CBE8",
    "base_delete": "#F28E2B",
    "ft_delete": "#FFBE7D",
    "base_sub": "#E15759",
    "ft_sub": "#FF9D9A",
}

for i, model in enumerate(models):

    subset = df[df["model_name"] == model]

    base = subset[subset["type"] == "base"].iloc[0]
    ft   = subset[subset["type"] == "finetuned"].iloc[0]

    values = [
        base["pct_insertions"],
        ft["pct_insertions"],
        base["pct_deletions"],
        ft["pct_deletions"],
        base["pct_substitutions"],
        ft["pct_substitutions"],
    ]

    offsets = np.array([
        -2.5*width,
        -1.5*width,
        -0.5*width,
         0.5*width,
         1.5*width,
         2.5*width
    ])

    labels = [
        "Ins Base",
        "Ins FT",
        "Del Base",
        "Del FT",
        "Sub Base",
        "Sub FT"
    ]

    colors_list = [
        colors["base_insert"],
        colors["ft_insert"],
        colors["base_delete"],
        colors["ft_delete"],
        colors["base_sub"],
        colors["ft_sub"],
    ]

    for j in range(6):
        bar = plt.bar(
            x[i] + offsets[j],
            values[j],
            width,
            color=colors_list[j],
            label=labels[j] if i == 0 else ""
        )

        # nilai di atas bar
        plt.text(
            x[i] + offsets[j],
            values[j] + 0.3,
            f"{values[j]:.2f}",
            ha='center',
            fontsize=8
        )

plt.xticks(x, models)
plt.ylabel("Persentase Kesalahan (%)")
plt.title("Perbandingan IDS Seluruh Model ASR", fontsize=14, fontweight="bold")
plt.legend(ncol=3)
plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

plt.savefig(
    "/kaggle/working/grafik_ids_semua_model.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.legend(ncol=3)
plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()

plt.savefig(
    "/kaggle/working/grafik_ids_semua_model.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()